# 14 · Modelado y validación con 17 partidos (experimento de ampliación)

Pregunta: ¿el modelo de los 5 core **transfiere** a otros partidos (E1) y cuánto mejora **reentrenar** con todos (E2)?

**Diseño (fijado antes de correr)**

- **Validación:** walk-forward como el notebook 04: entrena con campañas `< t`, predice `t`, para `t` = 2016 … 2024 (9 años).
- **E1 (transferencia):** entrena solo con los 5 core y predice los 17 partidos cada año. Las filas de los 5 core son la **referencia** (core → core); las de los 12 nuevos miden la transferencia.
- **E2 (reentrenar):** entrena y testea con los 17.
- **Modelo principal:** el Random Forest del modelo v2 adoptado, sin retunear: `RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)`, `StandardScaler` ajustado en el train, un modelo por ventana, features acumuladas (16 / 32 / 48 / 64 / 80).
- **Modelo desafiante:** XGBoost con la configuración del notebook 07, tal cual: `n_estimators=100, max_depth=3, learning_rate=0.05, reg_alpha=0, reg_lambda=1`.
- **Sin** `partido`, `zona`, `grupo`, `grupo_celda` ni superficies como features: solo para reportar.
- **Chequeo duro:** la referencia debe reproducir los MAE del notebook 04 (5 ventanas, tolerancia 0,01). Si no, el notebook se detiene.

**⚠ Sobre XGBoost:** su configuración se **tuneó con los años de test en el notebook 07** (se probaron 6 configuraciones y esta ganó por MAE en el walk-forward 2016-2024, con las features v1 de `oct_ene`). Tiene entonces una ventaja de selección sobre el test: una victoria de XGBoost pesa menos que una derrota.

**Regla de decisión (fijada antes de correr).** XGBoost reemplaza al Random Forest solo si, en **E2**:

- (a) gana en al menos **6 de los 9** años de test (MAE por año sobre los 17 partidos), y
- (b) en **cada una de las 4 zonas** (core, nucleo, transicion, oeste) `|sesgo XGB| <= |sesgo RF| + 0,2 qq/ha`.

Ventana principal: **`oct_nov`**. `oct_ene` es el control: si las dos ventanas se contradicen, XGBoost no se adopta.

**Convenciones.** Sesgo = predicho − real (positivo = sobreestima). Acierto direccional como en el 04: el signo de (real − promedio histórico del partido) coincide con el de (pred − promedio histórico). Semilla 42 en todo.

In [1]:
# Paths del proyecto: centralizados en src/config.py
import sys
from pathlib import Path

_raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if _raiz is None:
    raise FileNotFoundError("No se encontró src/config.py: abrí el notebook desde la carpeta del proyecto")
if str(_raiz / "src") not in sys.path:
    sys.path.insert(0, str(_raiz / "src"))
from config import PROJECT_ROOT, DATA_RAW, DATA_PROCESSED, OUTPUTS

In [2]:
import numpy as np
import pandas as pd
import warnings
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
warnings.filterwarnings('ignore')

try:
    from xgboost import XGBRegressor
except ImportError as e:
    raise ImportError("Falta xgboost: corré  pip install xgboost  y reiniciá el kernel") from e

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)

# ---- Parámetros fijados de antemano ----
SEED = 42
PRIMER_TEST, ULTIMO_TEST = 2016, 2024
VENTANAS = ["oct_nov", "oct_dic", "oct_ene", "oct_feb", "oct_mar"]
VENTANA_PRINCIPAL, VENTANA_CONTROL = "oct_nov", "oct_ene"
MIN_ANIOS_XGB = 6        # XGBoost debe ganar en >= 6 de 9 años de test (E2)
TOL_SESGO = 0.2          # |sesgo XGB| <= |sesgo RF| + 0.2 qq/ha en cada zona
TOL_NB04 = 0.01          # tolerancia del chequeo duro contra el notebook 04

# Random Forest del modelo v2 adoptado (notebooks 04 / 05): sin retunear
def rf_modelo(seed=SEED):
    return RandomForestRegressor(n_estimators=100, max_depth=5, random_state=seed)

# XGBoost del notebook 07, tal cual (tuneado con los años de test en el 07)
def xgb_modelo(seed=SEED):
    return XGBRegressor(n_estimators=100, max_depth=3, learning_rate=0.05,
                        reg_alpha=0, reg_lambda=1, random_state=seed, verbosity=0)

MODELOS = {"RF": rf_modelo, "XGB": xgb_modelo}

# MAE del Random Forest en el notebook 04 (walk-forward 2016-2024, 5 core), por ventana
MAE_NB04 = {"oct_nov": 2.42, "oct_dic": 2.64, "oct_ene": 3.01, "oct_feb": 2.94, "oct_mar": 3.02}

print("xgboost listo")

xgboost listo


## 1. Datos y features

In [3]:
df = pd.read_csv(DATA_PROCESSED / "dataset_modelado_ampliacion.csv")

# Metadatos y target: NO son features
META_COLS = ["campana", "partido", "grupo", "zona", "grupo_celda",
             "rinde_qq_ha", "superficie_sembrada_ha", "superficie_cosechada_ha"]

def features_ventana(ventana):
    '''Features acumuladas hasta esa ventana (misma regla que el notebook 04).'''
    disponibles = VENTANAS[:VENTANAS.index(ventana) + 1]
    return [c for c in df.columns if c not in META_COLS and any(c.endswith(v) for v in disponibles)]

assert len(df) == 425
CORE = df.loc[df["grupo"] == "core", "partido"].unique().tolist()
NUEVOS = df.loc[df["grupo"] == "ampliacion", "partido"].unique().tolist()
TODOS = CORE + NUEVOS
assert len(CORE) == 5 and len(NUEVOS) == 12
ORDEN_ZONAS = ["core", "nucleo", "transicion", "oeste"]

for v, n_esperado in zip(VENTANAS, [16, 32, 48, 64, 80]):
    f = features_ventana(v)
    assert len(f) == n_esperado, (v, len(f))
    assert not set(f) & set(META_COLS)
    print(f"  {v}: {len(f)} features")
print("Zonas:", df.drop_duplicates("partido").groupby("zona")["partido"].apply(list).to_dict())

  oct_nov: 16 features
  oct_dic: 32 features
  oct_ene: 48 features
  oct_feb: 64 features
  oct_mar: 80 features
Zonas: {'core': ['General Arenales', 'General Pinto', 'Junín', 'Leandro N. Alem', 'Lincoln'], 'nucleo': ['Colón', 'Pergamino', 'Rojas', 'Salto'], 'oeste': ['Carlos Tejedor', 'General Villegas', 'Pehuajó', 'Trenque Lauquen'], 'transicion': ['9 de Julio', 'Bragado', 'Chacabuco', 'General Viamonte']}


## 2. Walk-forward y métricas

In [4]:
def walk_forward(modelo_fn, features, partidos_train):
    '''Entrena con campañas < t (solo partidos_train) y predice los 17 partidos en t.'''
    # Promedio histórico del partido hasta t-1 (baseline y referencia del acierto direccional)
    salida = []
    for t in range(PRIMER_TEST, ULTIMO_TEST + 1):
        train = df[(df["campana"] < t) & (df["partido"].isin(partidos_train))]
        test = df[df["campana"] == t]
        assert len(test) == 17

        sc = StandardScaler()
        X_tr = sc.fit_transform(train[features])
        X_te = sc.transform(test[features])
        modelo = modelo_fn().fit(X_tr, train["rinde_qq_ha"].values)

        hist = df[df["campana"] < t].groupby("partido")["rinde_qq_ha"].mean()
        salida.append(pd.DataFrame({
            "campana": t,
            "partido": test["partido"].values,
            "grupo": test["grupo"].values,
            "zona": test["zona"].values,
            "rinde_real": test["rinde_qq_ha"].values,
            "rinde_pred": modelo.predict(X_te),
            "naive": test["partido"].map(hist).values,
            "sup_sembrada": test["superficie_sembrada_ha"].values,
        }))
    return pd.concat(salida, ignore_index=True)


def metricas(d):
    err = d["rinde_pred"] - d["rinde_real"]
    dir_ok = np.sign(d["rinde_real"] - d["naive"]) == np.sign(d["rinde_pred"] - d["naive"])
    return pd.Series({
        "n": len(d),
        "MAE": err.abs().mean(),
        "sesgo": err.mean(),
        "R2": r2_score(d["rinde_real"], d["rinde_pred"]) if len(d) > 1 else np.nan,
        "acierto_dir": dir_ok.mean(),
        "MAE_naive": (d["rinde_real"] - d["naive"]).abs().mean(),
    })


def tabla_zona(d):
    filas = {}
    for z in ORDEN_ZONAS:
        s = d[d["zona"] == z]
        if len(s):
            filas[z] = metricas(s)
    nu = d[d["grupo"] == "ampliacion"]
    if len(nu):
        filas["nuevos (12)"] = metricas(nu)
    filas[f"total ({d['partido'].nunique()})"] = metricas(d)
    return pd.DataFrame(filas).T


def tabla_partido(d):
    filas = {}
    for p in TODOS:
        s = d[d["partido"] == p]
        if len(s):
            m = metricas(s)
            m["zona"] = s["zona"].iloc[0]
            filas[p] = m
    return pd.DataFrame(filas).T

print("Funciones definidas")

Funciones definidas


## 3. Chequeo duro: la referencia (5 core → 5 core) debe reproducir el notebook 04

Random Forest entrenado solo con los 5 core, evaluado solo sobre las filas de los 5 core, en las 5 ventanas. Si algún MAE difiere en más de 0,01 del notebook 04, **el notebook se detiene**.

In [5]:
filas_chk = []
for v in VENTANAS:
    d = walk_forward(rf_modelo, features_ventana(v), CORE)
    ref = d[d["grupo"] == "core"]
    assert len(ref) == 45
    mae = metricas(ref)["MAE"]
    filas_chk.append({"ventana": v, "MAE_14": round(mae, 3), "MAE_nb04": MAE_NB04[v],
                      "dif": round(mae - MAE_NB04[v], 3), "ok": abs(mae - MAE_NB04[v]) <= TOL_NB04})
chk = pd.DataFrame(filas_chk)
display(chk)
if not chk["ok"].all():
    raise AssertionError("La referencia NO reproduce el notebook 04 (tolerancia 0.01). "
                         "Frená y revisá antes de seguir (versión de sklearn, dataset, features).")
print("OK: la referencia reproduce los MAE del notebook 04 en las 5 ventanas.")

,ventana,MAE_14,MAE_nb04,dif,ok
0,oct_nov,2.425,2.42,0.005,True
1,oct_dic,2.640,2.64,0.000,True
2,oct_ene,3.015,3.01,0.005,True
3,oct_feb,2.939,2.94,-0.001,True
4,oct_mar,3.018,3.02,-0.002,True


OK: la referencia reproduce los MAE del notebook 04 en las 5 ventanas.


## 4. Corridas: E1 y E2 × RF y XGBoost × 5 ventanas

In [6]:
EXPERIMENTOS = {"E1": CORE, "E2": TODOS}   # partidos de entrenamiento

partes = []
for v in VENTANAS:
    feats = features_ventana(v)
    for exp, partidos_train in EXPERIMENTOS.items():
        for nombre, fn in MODELOS.items():
            d = walk_forward(fn, feats, partidos_train)
            d.insert(0, "ventana", v)
            d.insert(0, "modelo", nombre)
            d.insert(0, "exp", exp)
            partes.append(d)
P = pd.concat(partes, ignore_index=True)
print(f"Predicciones: {len(P)} filas = 2 exp × 2 modelos × 5 ventanas × 9 años × 17 partidos")
assert len(P) == 2 * 2 * 5 * 9 * 17

def sel(exp, modelo, ventana, grupo=None):
    d = P[(P["exp"] == exp) & (P["modelo"] == modelo) & (P["ventana"] == ventana)]
    if grupo is not None:
        d = d[d["grupo"] == grupo]
    return d.sort_values(["campana", "partido"]).reset_index(drop=True)

Predicciones: 3060 filas = 2 exp × 2 modelos × 5 ventanas × 9 años × 17 partidos


## 5. Resultados por zona (T1) — Random Forest

Tres filas de modelos: **Ref** (core → core, E1 sobre los 5 core), **E1** (entrenado con los 5 core; las zonas nucleo / transicion / oeste miden la transferencia) y **E2** (entrenado con los 17).

In [7]:
def mostrar_t1(ventana, modelo="RF"):
    print(f"\n{'='*30} {modelo} · {ventana} {'='*30}")
    e1 = sel("E1", modelo, ventana)
    e2 = sel("E2", modelo, ventana)
    ref = tabla_zona(e1[e1["grupo"] == "core"])
    print("\nRef (5 core entrenan, se evalúa sobre los 5 core):")
    display(ref.round(3))
    print("E1 (5 core entrenan, se evalúa sobre los 17):")
    display(tabla_zona(e1).round(3))
    print("E2 (17 entrenan, se evalúa sobre los 17):")
    display(tabla_zona(e2).round(3))

mostrar_t1(VENTANA_PRINCIPAL)
mostrar_t1(VENTANA_CONTROL)


============================== RF · oct_nov ==============================

Ref (5 core entrenan, se evalúa sobre los 5 core):


,n,MAE,sesgo,R2,acierto_dir,MAE_naive
core,45.0,2.425,0.264,0.718,0.689,4.036
total (5),45.0,2.425,0.264,0.718,0.689,4.036


E1 (5 core entrenan, se evalúa sobre los 17):


,n,MAE,sesgo,R2,acierto_dir,MAE_naive
core,45.0,2.425,0.264,0.718,0.689,4.036
nucleo,36.0,5.063,-4.622,0.199,0.528,5.244
transicion,36.0,3.446,-0.289,0.551,0.694,4.563
oeste,36.0,6.243,1.431,-0.407,0.472,5.830
nuevos (12),108.0,4.917,-1.160,0.212,0.565,5.212
total (17),153.0,4.184,-0.741,0.343,0.601,4.866


E2 (17 entrenan, se evalúa sobre los 17):


,n,MAE,sesgo,R2,acierto_dir,MAE_naive
core,45.0,3.913,0.354,0.379,0.711,4.036
nucleo,36.0,5.739,-3.473,0.025,0.639,5.244
transicion,36.0,4.464,0.869,0.256,0.667,4.563
oeste,36.0,6.176,-0.784,-0.253,0.472,5.830
nuevos (12),108.0,5.460,-1.129,0.101,0.593,5.212
total (17),153.0,5.005,-0.693,0.178,0.627,4.866


... (2 outputs left out)

E1 (5 core entrenan, se evalúa sobre los 17):


,n,MAE,sesgo,R2,acierto_dir,MAE_naive
core,45.0,3.015,0.943,0.609,0.689,4.036
nucleo,36.0,4.155,-3.349,0.496,0.528,5.244
transicion,36.0,2.839,0.408,0.663,0.778,4.563
oeste,36.0,4.948,-0.226,0.165,0.611,5.830
nuevos (12),108.0,3.981,-1.056,0.500,0.639,5.212
total (17),153.0,3.697,-0.468,0.532,0.654,4.866


E2 (17 entrenan, se evalúa sobre los 17):


,n,MAE,sesgo,R2,acierto_dir,MAE_naive
core,45.0,3.768,1.005,0.350,0.711,4.036
nucleo,36.0,3.733,-1.551,0.584,0.806,5.244
transicion,36.0,3.670,0.716,0.412,0.806,4.563
oeste,36.0,5.324,-1.738,-0.031,0.694,5.830
nuevos (12),108.0,4.242,-0.858,0.395,0.769,5.212
total (17),153.0,4.103,-0.310,0.391,0.752,4.866


## 6. Resultados por partido (T2) — Random Forest

In [8]:
for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    print(f"\n{'='*20} Por partido · RF · {v} {'='*20}")
    t_e1 = tabla_partido(sel("E1", "RF", v))
    t_e2 = tabla_partido(sel("E2", "RF", v))
    cols = ["MAE", "sesgo", "R2", "acierto_dir"]
    comb = pd.concat({"E1": t_e1[cols].astype(float), "E2": t_e2[cols].astype(float)}, axis=1)
    comb.index = pd.MultiIndex.from_arrays([t_e1["zona"].values, comb.index], names=["zona", "partido"])
    display(comb.round(3))


==================== Por partido · RF · oct_nov ====================


E1                               E2                          
                               MAE  sesgo     R2 acierto_dir    MAE  sesgo     R2 acierto_dir
zona       partido                                                                           
core       General Arenales  3.461  0.420  0.643       0.556  4.576 -0.787  0.444       0.778
           General Pinto     1.802  0.883  0.660       0.556  3.569  1.004 -0.297       0.556
           Junín             3.144 -2.533  0.686       0.778  3.819 -0.874  0.528       0.889
           Leandro N. Alem   2.050  1.301  0.745       0.778  4.122  1.069  0.199       0.667
           Lincoln           1.668  1.249  0.751       0.778  3.478  1.359  0.054       0.667
transicion 9 de Julio        3.684  1.578  0.385       0.778  4.319  2.717  0.094       0.556
           Bragado           3.433  0.613  0.615       0.778  4.636  2.304  0.229       0.667
oeste      Carlos Tejedor    7.139  1.781 -0.009       0.444  6.646 -0.242  0.029       0.667
transicion Chacabuco         4.513 -3.956  0.465       0.667  5.333 -2.456  0.334       0.778
nucleo     Colón             5.229 -4.787  0.121       0.556  6.110 -4.295 -0.221       0.667
transicion General Viamonte  2.153  0.609  0.763       0.556  3.567  0.911  0.105       0.667
oeste      General Villegas  4.445  1.662 -0.922       0.444  4.899 -1.072 -1.455       0.333
           Pehuajó           6.107 -0.071 -0.217       0.556  6.805 -0.689 -0.268       0.444
nucleo     Pergamino         5.409 -4.910  0.111       0.556  6.204 -3.959 -0.074       0.667
           Rojas             4.935 -4.221  0.400       0.444  5.099 -2.558  0.363       0.556
           Salto             4.679 -4.571  0.081       0.556  5.544 -3.079 -0.099       0.667
oeste      Trenque Lauquen   7.280  2.352 -1.349       0.444  6.353 -1.133 -0.438       0.444


==================== Por partido · RF · oct_ene ====================


E1                               E2                          
                               MAE  sesgo     R2 acierto_dir    MAE  sesgo     R2 acierto_dir
zona       partido                                                                           
core       General Arenales  3.986  1.153  0.540       0.556  4.544  2.396  0.327       0.556
           General Pinto     2.415  1.833  0.485       0.556  2.451  0.962  0.397       0.889
           Junín             3.670 -1.976  0.603       0.778  4.856 -1.704  0.364       0.667
           Leandro N. Alem   3.176  2.479  0.475       0.556  4.061  3.119  0.067       0.667
           Lincoln           1.829  1.227  0.762       1.000  2.927  0.252  0.229       0.778
transicion 9 de Julio        2.782  1.329  0.603       0.778  2.919  1.655  0.475       1.000
           Bragado           2.918  1.194  0.618       0.889  3.709  1.566  0.273       0.667
oeste      Carlos Tejedor    5.972 -0.320  0.299       0.667  6.716 -1.835 -0.010       0.444
transicion Chacabuco         3.699 -2.375  0.654       0.778  4.574 -0.723  0.474       0.889
nucleo     Colón             4.402 -3.428  0.452       0.556  3.830 -2.263  0.544       0.778
transicion General Viamonte  1.956  1.483  0.742       0.667  3.476  0.367  0.135       0.667
oeste      General Villegas  3.941  1.784 -0.670       0.556  3.591  0.790 -0.329       0.778
           Pehuajó           4.316 -0.623  0.385       0.778  4.877 -1.896  0.253       0.889
nucleo     Pergamino         4.478 -4.029  0.441       0.556  3.820 -2.942  0.600       1.000
           Rojas             3.614 -2.377  0.653       0.444  4.015 -0.198  0.608       0.667
           Salto             4.125 -3.563  0.365       0.556  3.266 -0.802  0.568       0.778
oeste      Trenque Lauquen   5.564 -1.743 -0.158       0.444  6.114 -4.012 -0.487       0.667

## 7. Comparación sobre las mismas filas (T3)

- **12 nuevos:** E1 (transferencia) vs E2 (reentrenado con los 17), 108 predicciones.
- **5 core:** Ref (modelo de los 5 core) vs E2, 45 predicciones. Mide si sumar partidos ayuda o perjudica a los core.

In [9]:
filas_t3 = []
for v in VENTANAS:
    e1, e2 = sel("E1", "RF", v), sel("E2", "RF", v)
    for etiqueta, grupo, nombre_a in [("12 nuevos: E1 vs E2", "ampliacion", "E1"), ("5 core: Ref vs E2", "core", "Ref")]:
        a, b = e1[e1["grupo"] == grupo], e2[e2["grupo"] == grupo]
        assert (a["partido"].values == b["partido"].values).all() and (a["campana"].values == b["campana"].values).all()
        ma, mb = metricas(a), metricas(b)
        filas_t3.append({"ventana": v, "filas": etiqueta, "n": int(ma["n"]),
                         "MAE_A": ma["MAE"], "MAE_E2": mb["MAE"], "dif_MAE(E2-A)": mb["MAE"] - ma["MAE"],
                         "sesgo_A": ma["sesgo"], "sesgo_E2": mb["sesgo"],
                         "R2_A": ma["R2"], "R2_E2": mb["R2"],
                         "dir_A": ma["acierto_dir"], "dir_E2": mb["acierto_dir"], "MAE_naive": ma["MAE_naive"]})
T3 = pd.DataFrame(filas_t3)
print("A = E1 para '12 nuevos' y Ref para '5 core'. dif_MAE < 0 significa que E2 mejora.")
display(T3.round(3))

A = E1 para '12 nuevos' y Ref para '5 core'. dif_MAE < 0 significa que E2 mejora.


,ventana,filas,n,MAE_A,MAE_E2,dif_MAE(E2-A),sesgo_A,sesgo_E2,R2_A,R2_E2,dir_A,dir_E2,MAE_naive
0,oct_nov,12 nuevos: E1 vs E2,108,4.917,5.460,0.542,-1.160,-1.129,0.212,0.101,0.565,0.593,5.212
1,oct_nov,5 core: Ref vs E2,45,2.425,3.913,1.488,0.264,0.354,0.718,0.379,0.689,0.711,4.036
2,oct_dic,12 nuevos: E1 vs E2,108,4.621,4.759,0.138,-1.220,-1.021,0.320,0.293,0.583,0.639,5.212
3,oct_dic,5 core: Ref vs E2,45,2.640,3.477,0.837,0.785,0.949,0.695,0.484,0.689,0.711,4.036
4,oct_ene,12 nuevos: E1 vs E2,108,3.981,4.242,0.262,-1.056,-0.858,0.500,0.395,0.639,0.769,5.212
5,oct_ene,5 core: Ref vs E2,45,3.015,3.768,0.753,0.943,1.005,0.609,0.350,0.689,0.711,4.036
6,oct_feb,12 nuevos: E1 vs E2,108,3.714,3.593,-0.121,-1.301,-0.900,0.566,0.559,0.676,0.806,5.212
7,oct_feb,5 core: Ref vs E2,45,2.939,3.544,0.605,0.719,1.032,0.621,0.435,0.711,0.689,4.036
8,oct_mar,12 nuevos: E1 vs E2,108,3.500,3.118,-0.381,-1.083,-0.972,0.606,0.670,0.704,0.852,5.212
9,oct_mar,5 core: Ref vs E2,45,3.018,3.167,0.149,0.805,0.876,0.589,0.499,0.733,0.689,4.036


## 8. Rinde regional predicho vs real (T4)

Promedio regional por campaña, en versión **simple** (media de partidos) y **ponderada** por superficie sembrada del año de test. Aviso: en un uso real la superficie del año sería una estimación; acá se usa la observada.

In [10]:
def regional(d):
    filas = []
    for t, x in d.groupby("campana"):
        w = x["sup_sembrada"].values
        filas.append({"campana": t,
                      "real_simple": x["rinde_real"].mean(), "pred_simple": x["rinde_pred"].mean(),
                      "real_pond": np.average(x["rinde_real"], weights=w), "pred_pond": np.average(x["rinde_pred"], weights=w)})
    r = pd.DataFrame(filas).set_index("campana")
    r["err_simple"] = r["pred_simple"] - r["real_simple"]
    r["err_pond"] = r["pred_pond"] - r["real_pond"]
    return r

def resumen_regional(r):
    return pd.Series({"MAE_simple": r["err_simple"].abs().mean(), "sesgo_simple": r["err_simple"].mean(),
                      "MAE_pond": r["err_pond"].abs().mean(), "sesgo_pond": r["err_pond"].mean()})

SUBCONJUNTOS = {"total (17)": lambda d: d,
                "core (5)": lambda d: d[d["grupo"] == "core"],
                "nuevos (12)": lambda d: d[d["grupo"] == "ampliacion"],
                **{z: (lambda d, z=z: d[d["zona"] == z]) for z in ["nucleo", "transicion", "oeste"]}}

REGIONAL = {}
for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    for exp in ["E1", "E2"]:
        d = sel(exp, "RF", v)
        for nombre, f in SUBCONJUNTOS.items():
            REGIONAL[(v, exp, nombre)] = regional(f(d))

for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    print(f"\n{'='*20} Resumen regional (RF, {v}): error del promedio regional, 9 años {'='*20}")
    res = pd.DataFrame({(exp, nombre): resumen_regional(REGIONAL[(v, exp, nombre)])
                        for exp in ["E1", "E2"] for nombre in SUBCONJUNTOS}).T
    display(res.round(3))

print("\nDetalle por año (RF, oct_nov): total (17) en E2 y nuevos (12) en E1 y E2")
for clave in [(VENTANA_PRINCIPAL, "E2", "total (17)"), (VENTANA_PRINCIPAL, "E1", "nuevos (12)"), (VENTANA_PRINCIPAL, "E2", "nuevos (12)")]:
    print("\n", clave)
    display(REGIONAL[clave].round(2))


==================== Resumen regional (RF, oct_nov): error del promedio regional, 9 años ====================


MAE_simple  sesgo_simple  MAE_pond  sesgo_pond
E1 total (17)        3.071        -0.741     3.065      -0.522
   core (5)          1.797         0.264     1.677       0.342
   nuevos (12)       3.815        -1.160     3.638      -0.801
   nucleo            4.857        -4.622     4.924      -4.658
   transicion        2.709        -0.289     2.881      -0.102
   oeste             4.685         1.431     4.295       1.365
E2 total (17)        4.187        -0.693     4.141      -0.647
   core (5)          3.512         0.354     3.370       0.523
   nuevos (12)       4.725        -1.129     4.587      -1.027
   nucleo            5.459        -3.473     5.486      -3.450
   transicion        3.472         0.869     3.623       1.079
   oeste             5.760        -0.784     5.186      -0.950


==================== Resumen regional (RF, oct_ene): error del promedio regional, 9 años ====================


MAE_simple  sesgo_simple  MAE_pond  sesgo_pond
E1 total (17)        2.240        -0.468     2.133      -0.367
   core (5)          2.064         0.943     1.861       0.911
   nuevos (12)       2.793        -1.056     2.646      -0.779
   nucleo            3.453        -3.349     3.508      -3.438
   transicion        2.192         0.408     2.343       0.471
   oeste             3.465        -0.226     3.133       0.132
E2 total (17)        3.000        -0.310     2.894      -0.346
   core (5)          2.960         1.005     2.800       0.796
   nuevos (12)       3.086        -0.858     2.983      -0.715
   nucleo            3.052        -1.551     3.010      -1.669
   transicion        2.814         0.716     2.842       0.847
   oeste             4.438        -1.738     3.715      -1.259


Detalle por año (RF, oct_nov): total (17) en E2 y nuevos (12) en E1 y E2

 ('oct_nov', 'E2', 'total (17)')


,real_simple,pred_simple,real_pond,pred_pond,err_simple,err_pond
campana,,,,,,
2016,36.82,32.88,35.88,32.75,-3.94,-3.14
2017,28.02,33.02,27.52,32.89,5.00,5.36
2018,37.71,34.42,37.41,34.40,-3.29,-3.00
2019,34.32,33.98,34.12,33.49,-0.35,-0.63
2020,29.07,32.14,28.91,31.90,3.07,2.98
2021,31.35,25.11,31.21,25.17,-6.24,-6.04
2022,17.56,25.22,17.83,25.21,7.66,7.38
2023,32.71,28.90,32.84,28.57,-3.81,-4.27
2024,32.44,28.10,32.30,27.84,-4.34,-4.46



 ('oct_nov', 'E1', 'nuevos (12)')


,real_simple,pred_simple,real_pond,pred_pond,err_simple,err_pond
campana,,,,,,
2016,37.17,33.00,36.22,32.76,-4.17,-3.46
2017,27.71,31.08,27.28,31.34,3.37,4.06
2018,39.27,33.23,38.59,33.21,-6.04,-5.38
2019,34.74,31.97,34.48,32.20,-2.77,-2.28
2020,28.89,32.77,28.75,33.16,3.87,4.42
2021,32.04,29.86,31.68,30.50,-2.18,-1.18
2022,18.09,22.79,18.28,22.57,4.70,4.29
2023,33.40,29.71,33.36,28.94,-3.69,-4.43
2024,32.80,29.27,32.56,29.31,-3.53,-3.25



 ('oct_nov', 'E2', 'nuevos (12)')


,real_simple,pred_simple,real_pond,pred_pond,err_simple,err_pond
campana,,,,,,
2016,37.17,32.79,36.22,32.69,-4.38,-3.53
2017,27.71,32.61,27.28,32.61,4.91,5.33
2018,39.27,34.41,38.59,34.46,-4.86,-4.13
2019,34.74,33.29,34.48,32.88,-1.44,-1.60
2020,28.89,31.89,28.75,31.63,3.00,2.88
2021,32.04,25.82,31.68,25.74,-6.22,-5.94
2022,18.09,26.37,18.28,26.09,8.28,7.81
2023,33.40,28.66,33.36,28.26,-4.74,-5.11
2024,32.80,28.10,32.56,27.60,-4.70,-4.95


## 9. Random Forest vs XGBoost en E2 (T5)

MAE por año de test sobre los 17 partidos y diferencia apareada (XGB − RF; negativa = XGBoost mejor).

In [11]:
def mae_por_anio(d):
    ae = (d["rinde_pred"] - d["rinde_real"]).abs()
    return ae.groupby(d["campana"]).mean()

def comp_rf_xgb(ventana):
    rf, xg = sel("E2", "RF", ventana), sel("E2", "XGB", ventana)
    t = pd.DataFrame({"MAE_RF": mae_por_anio(rf), "MAE_XGB": mae_por_anio(xg)})
    t["dif_XGB-RF"] = t["MAE_XGB"] - t["MAE_RF"]
    t["gana"] = np.where(t["dif_XGB-RF"] < 0, "XGB", "RF")
    return t

for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    t = comp_rf_xgb(v)
    print(f"\n{'='*20} E2 · {v}: MAE por año de test {'='*20}")
    display(t.round(3))
    print(f"Años que gana XGBoost: {(t['gana'] == 'XGB').sum()} de {len(t)} | gana RF: {(t['gana'] == 'RF').sum()}")
    print(f"MAE total  RF: {metricas(sel('E2', 'RF', v))['MAE']:.3f}   XGB: {metricas(sel('E2', 'XGB', v))['MAE']:.3f}   "
          f"dif media por año (XGB-RF): {t['dif_XGB-RF'].mean():+.3f}")

print("\nTodas las ventanas (E2): MAE total y años que gana XGBoost")
filas = []
for v in VENTANAS:
    t = comp_rf_xgb(v)
    filas.append({"ventana": v, "MAE_RF": metricas(sel("E2", "RF", v))["MAE"], "MAE_XGB": metricas(sel("E2", "XGB", v))["MAE"],
                  "anios_gana_XGB": int((t["gana"] == "XGB").sum())})
display(pd.DataFrame(filas).round(3))


==================== E2 · oct_nov: MAE por año de test ====================


,MAE_RF,MAE_XGB,dif_XGB-RF,gana
campana,,,,
2016,4.940,4.330,-0.610,XGB
2017,6.563,6.521,-0.042,XGB
2018,4.716,4.522,-0.193,XGB
2019,2.474,2.945,0.471,RF
2020,3.068,2.783,-0.285,XGB
2021,6.412,5.957,-0.454,XGB
2022,7.657,9.340,1.683,RF
2023,3.881,4.543,0.661,RF
2024,5.332,2.795,-2.536,XGB


Años que gana XGBoost: 6 de 9 | gana RF: 3
MAE total  RF: 5.005   XGB: 4.860   dif media por año (XGB-RF): -0.145

==================== E2 · oct_ene: MAE por año de test ====================


,MAE_RF,MAE_XGB,dif_XGB-RF,gana
campana,,,,
2016,4.658,4.888,0.230,RF
2017,2.920,3.576,0.657,RF
2018,4.834,4.667,-0.167,XGB
2019,2.514,2.488,-0.026,XGB
2020,1.971,1.211,-0.760,XGB
2021,5.030,3.530,-1.500,XGB
2022,7.452,9.775,2.322,RF
2023,3.448,3.032,-0.415,XGB
2024,4.097,3.706,-0.391,XGB


Años que gana XGBoost: 6 de 9 | gana RF: 3
MAE total  RF: 4.103   XGB: 4.097   dif media por año (XGB-RF): -0.006

Todas las ventanas (E2): MAE total y años que gana XGBoost


,ventana,MAE_RF,MAE_XGB,anios_gana_XGB
0,oct_nov,5.005,4.860,6
1,oct_dic,4.382,4.533,4
2,oct_ene,4.103,4.097,6
3,oct_feb,3.578,3.550,6
4,oct_mar,3.132,3.326,5


## 10. Sesgo por zona y regla de decisión (T6)

Regla fijada de antemano: XGBoost reemplaza al Random Forest solo si, en E2, (a) gana en ≥ 6 de 9 años y (b) en cada zona `|sesgo XGB| <= |sesgo RF| + 0,2`. Si `oct_nov` y `oct_ene` se contradicen, no se adopta.

In [12]:
def veredicto(ventana):
    t = comp_rf_xgb(ventana)
    gana = int((t["gana"] == "XGB").sum())
    rf, xg = sel("E2", "RF", ventana), sel("E2", "XGB", ventana)
    zonas = pd.DataFrame({
        "sesgo_RF": {z: metricas(rf[rf["zona"] == z])["sesgo"] for z in ORDEN_ZONAS},
        "sesgo_XGB": {z: metricas(xg[xg["zona"] == z])["sesgo"] for z in ORDEN_ZONAS},
    })
    zonas["|XGB|-|RF|"] = zonas["sesgo_XGB"].abs() - zonas["sesgo_RF"].abs()
    zonas["cumple(<=0.2)"] = zonas["|XGB|-|RF|"] <= TOL_SESGO
    cond_a = gana >= MIN_ANIOS_XGB
    cond_b = bool(zonas["cumple(<=0.2)"].all())
    return {"ventana": ventana, "anios_gana_xgb": gana, "cond_a": cond_a, "zonas": zonas, "cond_b": cond_b, "pasa": cond_a and cond_b}

VEREDICTOS = {}
for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    r = veredicto(v)
    VEREDICTOS[v] = r
    print(f"\n{'='*20} {v} {'='*20}")
    print(f"(a) Gana en {r['anios_gana_xgb']} de 9 años (mínimo {MIN_ANIOS_XGB}): {'CUMPLE' if r['cond_a'] else 'NO cumple'}")
    display(r["zonas"].round(3))
    print(f"(b) Sesgo por zona dentro de la tolerancia: {'CUMPLE' if r['cond_b'] else 'NO cumple'}")
    print(f"=> {v}: {'XGBoost PASA la regla' if r['pasa'] else 'XGBoost NO pasa la regla'}")

print("\n" + "=" * 60)
pn, pc = VEREDICTOS[VENTANA_PRINCIPAL]["pasa"], VEREDICTOS[VENTANA_CONTROL]["pasa"]
if pn and pc:
    print("DECISIÓN: XGBoost pasa la regla en las dos ventanas -> reemplaza al Random Forest.")
elif pn != pc:
    print("DECISIÓN: las dos ventanas se contradicen -> XGBoost NO se adopta (se mantiene el Random Forest).")
else:
    print("DECISIÓN: XGBoost no pasa la regla en ninguna ventana -> se mantiene el Random Forest.")
print("Recordatorio: la configuración de XGBoost se tuneó con los años de test en el notebook 07.")


==================== oct_nov ====================
(a) Gana en 6 de 9 años (mínimo 6): CUMPLE


,sesgo_RF,sesgo_XGB,|XGB|-|RF|,cumple(<=0.2)
core,0.354,1.247,0.893,False
nucleo,-3.473,-1.961,-1.512,True
transicion,0.869,1.154,0.285,False
oeste,-0.784,-1.128,0.343,False


(b) Sesgo por zona dentro de la tolerancia: NO cumple
=> oct_nov: XGBoost NO pasa la regla

==================== oct_ene ====================
(a) Gana en 6 de 9 años (mínimo 6): CUMPLE


,sesgo_RF,sesgo_XGB,|XGB|-|RF|,cumple(<=0.2)
core,1.005,1.186,0.181,True
nucleo,-1.551,-1.092,-0.459,True
transicion,0.716,0.939,0.222,False
oeste,-1.738,-1.662,-0.076,True


(b) Sesgo por zona dentro de la tolerancia: NO cumple
=> oct_ene: XGBoost NO pasa la regla

DECISIÓN: XGBoost no pasa la regla en ninguna ventana -> se mantiene el Random Forest.
Recordatorio: la configuración de XGBoost se tuneó con los años de test en el notebook 07.


## 11. Sensibilidad a la semilla (solo informativa, no se usa para decidir)

MAE total en E2 con 10 semillas, para RF y XGBoost, en las dos ventanas. XGBoost con estos parámetros no usa submuestreo, por lo que se espera que casi no varíe con la semilla.

In [13]:
filas = []
for v in [VENTANA_PRINCIPAL, VENTANA_CONTROL]:
    feats = features_ventana(v)
    for nombre, fn in MODELOS.items():
        maes = []
        for seed in range(10):
            d = walk_forward(lambda s=seed, f=fn: f(s), feats, TODOS)
            maes.append(metricas(d)["MAE"])
        filas.append({"ventana": v, "modelo": nombre, "MAE_media": np.mean(maes), "std": np.std(maes),
                      "min": np.min(maes), "max": np.max(maes)})
display(pd.DataFrame(filas).round(3))

,ventana,modelo,MAE_media,std,min,max
0,oct_nov,RF,4.874,0.106,4.764,5.061
1,oct_nov,XGB,4.860,0.000,4.860,4.860
2,oct_ene,RF,4.135,0.042,4.088,4.214
3,oct_ene,XGB,4.097,0.000,4.097,4.097


## 12. Guardar resultados

In [14]:
OUT = OUTPUTS / "ampliacion"
OUT.mkdir(parents=True, exist_ok=True)

P.to_csv(OUT / "predicciones_nb14.csv", index=False)

zonas_all = []
for exp in ["E1", "E2"]:
    for modelo in MODELOS:
        for v in VENTANAS:
            t = tabla_zona(sel(exp, modelo, v)).reset_index().rename(columns={"index": "grupo_filas"})
            t.insert(0, "ventana", v); t.insert(0, "modelo", modelo); t.insert(0, "exp", exp)
            zonas_all.append(t)
pd.concat(zonas_all).to_csv(OUT / "metricas_zona_nb14.csv", index=False)

partidos_all = []
for exp in ["E1", "E2"]:
    for modelo in MODELOS:
        for v in VENTANAS:
            t = tabla_partido(sel(exp, modelo, v)).reset_index().rename(columns={"index": "partido"})
            t.insert(0, "ventana", v); t.insert(0, "modelo", modelo); t.insert(0, "exp", exp)
            partidos_all.append(t)
pd.concat(partidos_all).to_csv(OUT / "metricas_partido_nb14.csv", index=False)

pd.concat({v: comp_rf_xgb(v) for v in VENTANAS}, names=["ventana"]).to_csv(OUT / "rf_vs_xgb_e2_nb14.csv")
T3.to_csv(OUT / "comparacion_mismas_filas_nb14.csv", index=False)
pd.concat({"|".join(k): r for k, r in REGIONAL.items()}, names=["ventana|exp|subconjunto"]).to_csv(OUT / "regional_nb14.csv")
chk.to_csv(OUT / "chequeo_referencia_nb04_nb14.csv", index=False)

print("Archivos en", OUT)
for f in sorted(OUT.glob("*_nb14.csv")):
    print("  -", f.name)

  - chequeo_referencia_nb04_nb14.csv
  - comparacion_mismas_filas_nb14.csv
  - metricas_partido_nb14.csv
  - metricas_zona_nb14.csv
  - predicciones_nb14.csv
  - regional_nb14.csv
  - rf_vs_xgb_e2_nb14.csv
